In [ ]:
import pickle
import re
from tqdm import tqdm
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from openpyxl import load_workbook

import geopandas as gpd
from shapely.geometry import Point, Polygon

import osmnx as ox
import networkx as nx

import plotly.express as px
import matplotlib.pyplot as plt


In [ ]:
with open("properties_scraped_Portuense.pkl", "rb") as fp:
    test = pickle.load(fp)

with open("properties_scraped_Gianicolense.pkl", "rb") as fp:
    test2 = pickle.load(fp)

with open("properties_scraped_Ostiense.pkl", "rb") as fp:
    test3 = pickle.load(fp)

for k,v in test2.items():
    test[k] = v
for k,v in test3.items():
    test[k] = v

booking_df = pd.DataFrame.from_dict(test).transpose()

In [ ]:
# Your GeoJSON data (the coordinates you provided)
geojson_data_villa_bonelli = [
    [12.460386688914582, 41.857304530879475],
    [12.455066080349411, 41.85592945863141],
    [12.443687920522649, 41.849415798400486],
    [12.447134060296833, 41.84003115614769],
    [12.450581757287921, 41.83893886798171],
    [12.454031337926239, 41.84120468429387],
    [12.457611907515002, 41.83952354610173],
    [12.464108153304153, 41.845352679035045],
    [12.470860067635158, 41.847036779217376],
    [12.469546162015718, 41.849302756347896],
    [12.46092004198593, 41.85276277745561],
    [12.460386688914582, 41.857304530879475]
]

# Coordinates from your GeoJSON
geojson_data_marconi_staz_trastevere = [
        [
            12.464677176611389, 41.86042593607078
        ],
        [
            12.47016889113172, 41.86042593607078
        ],
        [
            12.475660605653218, 41.86212286110697
        ],
        [
            12.476595365571967, 41.863602194909475
        ],
        [
            12.474316888270579, 41.86673478880235
        ],
        [
            12.47110365105047, 41.8729560131942
        ],
        [
            12.468416216285107, 41.8729560131942
        ],
        [
            12.466020893994028, 41.87343454383847
        ],
        [
            12.464501909126852, 41.872782001162705
        ],
        [
            12.461054981927305, 41.872694994968896
        ],
        [
            12.45965284204982, 41.87143339185502
        ],
        [
            12.461113404422179, 41.86629971549547
        ],
        [
            12.461931319351208, 41.86142669218094
        ],
        [
            12.464677176611389, 41.86042593607078
        ]
]

geojson_data_Ostiense_Garbatella = [
        [
            12.47244425099828, 41.87237509020889
        ],
        [
            12.47414713508681, 41.86779594899042
        ],
        [
            12.476796065891136, 41.866175558912914
        ],
        [
            12.477647507934506, 41.86385057967388
        ],
        [
            12.474998577131089, 41.85962312808857
        ],
        [
            12.492878860058198, 41.856663745684415
        ],
        [
            12.497798302979987, 41.86025726364383
        ],
        [
            12.49600081421994, 41.86716188819892
        ],
        [
            12.495811604876934, 41.870895711222545
        ],
        [
            12.49439253480287, 41.870895711222545
        ],
        [
            12.481242485454572, 41.876390374865565
        ],
        [
            12.47244425099828, 41.87237509020889
        ]
]

geojson_data_SanPaolo = [
        [
            12.475093181802578, 41.860750475838614
        ],
        [
            12.46837625012077, 41.85934128804513
        ],
        [
            12.463646016541986, 41.85955266819434
        ],
        [
            12.460807876395563, 41.85567725444986
        ],
        [
            12.463835225885816, 41.85194254265389
        ],
        [
            12.469984529537811, 41.84975798750642
        ],
        [
            12.47651225187576, 41.844895322735795
        ],
        [
            12.494960162832683, 41.85828437682119
        ],
        [
            12.475093181802578, 41.860750475838614
        ]
]

geojson_fullB_B_search =           [
            [
              12.47392668245078,
              41.841616935595255
            ],
            [
              12.497286285626132,
              41.859701689839994
            ],
            [
              12.497023252173761,
              41.86175863129978
            ],
            [
              12.496365668541529,
              41.871063015953894
            ],
            [
              12.481372761743188,
              41.876547071479564
            ],
            [
              12.476769676321425,
              41.86450111698716
            ],
            [
              12.474007825068895,
              41.86734142458545
            ],
            [
              12.469273222922226,
              41.87870139312599
            ],
            [
              12.473744791616525,
              41.88359730860154
            ],
            [
              12.45493789975356,
              41.88369522308474
            ],
            [
              12.450570363878512,
              41.88030010511295
            ],
            [
              12.44367468887134,
              41.8788095386183
            ],
            [
              12.443563468306792,
              41.85951184563402
            ],
            [
              12.440226851367783,
              41.8561983510929
            ],
            [
              12.445454217905933,
              41.840705486773516
            ],
            [
              12.450792805007524,
              41.83896541206343
            ],
            [
              12.453350877993842,
              41.8410369242105
            ],
            [
              12.457243597756587,
              41.84045690756918
            ],
            [
              12.46380561106912,
              41.84468262271929
            ],
            [
              12.47392668245078,
              41.841616935595255
            ]
          ]

geojson_BigArea = [
            [
              12.470894407472372,
              41.88521936912184
            ],
            [
              12.455740147496442,
              41.884928217462004
            ],
            [
              12.450069521183451,
              41.88019681704134
            ],
            [
              12.440390348682257,
              41.877357808621156
            ],
            [
              12.437652804945316,
              41.863961811286885
            ],
            [
              12.445572127899737,
              41.840730576689936
            ],
            [
              12.450851676536274,
              41.83861822789706
            ],
            [
              12.453589220274125,
              41.841021929672564
            ],
            [
              12.459846463102735,
              41.83883674997105
            ],
            [
              12.46473493406441,
              41.843644046815456
            ],
            [
              12.474596508799578,
              41.84148325790733
            ],
            [
              12.476046740377853,
              41.84465239000053
            ],
            [
              12.482234395113352,
              41.83975457424353
            ],
            [
              12.51404280773832,
              41.85315065112627
            ],
            [
              12.503407776161936,
              41.86647192362432
            ],
            [
              12.501667498267267,
              41.870791740364524
            ],
            [
              12.49605993616359,
              41.87122370598394
            ],
            [
              12.495383161426588,
              41.87986240535204
            ],
            [
              12.480687481429925,
              41.88792413792996
            ],
            [
              12.470894407472372,
              41.88521936912184
            ]
          ]


# Convert to Shapely Polygon (note: only one ring assumed here)
polygon_marconi_staz_trastevere = Polygon(geojson_data_marconi_staz_trastevere)

# Create a Polygon object using Shapely
polygon_villaBonelli = Polygon(geojson_data_villa_bonelli)

# Create a Polygon object using Shapely
polygon_Ostiense_Garbatella = Polygon(geojson_data_Ostiense_Garbatella)

# Create a Polygon object using Shapely
polygon_SanPaolo = Polygon(geojson_data_SanPaolo)

# Create a Polygon object using Shapely
polygon_fullB_B_search = Polygon(geojson_fullB_B_search)

# Create a Polygon object using Shapely
polygon_BigArea = Polygon(geojson_BigArea)

In [ ]:
# Combine all polygons into one GeoDataFrame
gdf_all = gpd.GeoDataFrame(geometry=[polygon_villaBonelli, 
                                     polygon_marconi_staz_trastevere, 
                                     polygon_Ostiense_Garbatella,
                                     polygon_SanPaolo,
                                     polygon_fullB_B_search,
                                     polygon_BigArea])

# Set a coordinate reference system (WGS84 - latitude/longitude)
gdf_all.set_crs(epsg=4326, inplace=True)

# Plot
gdf_all.plot(edgecolor='black', facecolor='lightblue', alpha=0.5)
plt.title("Polygons Plot")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

In [ ]:
# Function to check if a point is inside the polygon
def is_inside(lat, lon, polygon):
    point = Point(lon, lat)
    return polygon.contains(point)


In [ ]:
booking_df['is_inside_BigArea'] = booking_df.apply(lambda row: is_inside(row['latitude'], row['longitude'], polygon_BigArea), axis=1)


# Filter the DataFrame to keep only the points inside the polygon
filtered_booking_df = booking_df[booking_df['is_inside_BigArea']]

filtered_booking_df['latitude'] = pd.to_numeric(filtered_booking_df['latitude'], errors='coerce')
filtered_booking_df['longitude'] = pd.to_numeric(filtered_booking_df['longitude'], errors='coerce')

# Display the filtered DataFramedf['is_inside_villaBonelli']
print(filtered_booking_df)

In [ ]:
filtered_booking_df[filtered_booking_df['title']=='Casa McDoor']['calendar'].iloc[0]

In [ ]:
def get_hyperlinks(nome_file):
    wb = load_workbook(nome_file, data_only=False)
    ws = wb.active

    # Find the column index of the desired header
    target_header = "Link"
    col_index = None

    for cell in ws[1]:  # First row = header
        if cell.value == target_header:
            col_index = cell.column  # This gives the numeric column index (1 for A, 2 for B, etc.)
            break

    if col_index is None:
        raise ValueError(f"Header '{target_header}' not found!")

    # Now extract the formulas from that column
    hyperlinks = []
    for row in ws.iter_rows(min_row=2, min_col=col_index, max_col=col_index):
        formula = row[0].value
        if formula and formula.startswith('=HYPERLINK('):
            url = formula.replace('=HYPERLINK("', '').replace('")', '')
            hyperlinks.append(url)
        else:
            hyperlinks.append(None)
    return hyperlinks

lista_zone = [
              "Navigatori",
              "Garbatella",
              "Ostiense",
              "San Paolo",
              "Marconi",
              "Portuense",
              "Villa Bonelli",
              "Magliana",
              "Aventino",
              "San Saba - Caracalla",
              "Testaccio",
              "Trastevere",
              "Monteverde Vecchio",
              "Monteverde Nuovo",
              "Colle Oppio",
              "Colosseo - Fori Imperiali",
              "Monti"
]

immobiliare_df = pd.DataFrame(columns = ['Da vedere', 'ID', 'Prezzo old', 'Titolo', 'Link', 'Prezzo', 'Locali',
       'Dimensione', 'Bagni', 'Piano', 'Zona', 'Macro Zona',
       'Vendita Euro/m2 Macro Zona', 'Affitto Euro/m2 Macro Zona',
       'Rapporto Affitto / Vendita Macro Zona', 'Prezzo_Pulito',
       'Superficie_Pulita', 'Prezzo Euro/m2',
       'Differenza prezzo Euro/m2  - Vendita Euro/m2 Macro Zona',
       'Descrizione', 'Nuovo', 'Cambio Prezzo', 'Rimosso', 'Latitude',
       'Longitudes'])
for ind_zona, nome_zona in enumerate(lista_zone):
    nome_zona_pulito = re.sub(r"[^a-zA-Z0-9]+", ' ', nome_zona)
    print ("Zona: {}".format(nome_zona))
    try:
        nome_file = '../Visita_Lista_Annunci_Immobiliare_Case_2025_{}.xlsx'.format(nome_zona_pulito)
        visto_zona = pd.read_excel(nome_file, sheet_name = "Annunci Completo", index_col=0)
        hyperlinks = get_hyperlinks(nome_file)
        visto_zona['Link'] = hyperlinks
        visto_zona_filt = visto_zona[(visto_zona['Da vedere'].str.lower()=='si') & visto_zona['Latitude']!=0]
        if len(visto_zona_filt)>0:
            immobiliare_df = immobiliare_df.append(visto_zona_filt)
    except:
        print ("Zona: {} non esiste".format(nome_zona))
    finally:
        print("-"*80)

In [ ]:
immobiliare_df

In [ ]:
Counter(immobiliare_df['Zona'])

In [ ]:
# Step 1: Get bounding box of all coordinates to download a relevant road network
minx = min(immobiliare_df['Longitudes'].min(), filtered_booking_df['longitude'].min()) - 0.01
maxx = max(immobiliare_df['Longitudes'].max(), filtered_booking_df['longitude'].max()) + 0.01
miny = min(immobiliare_df['Latitude'].min(), filtered_booking_df['latitude'].min()) - 0.01
maxy = max(immobiliare_df['Latitude'].max(), filtered_booking_df['latitude'].max()) + 0.01

# Step 2: Download the street network
G = ox.graph_from_bbox(north=maxy, south=miny, east=maxx, west=minx, network_type='drive')



In [ ]:
# Step 3: Convert coordinates to nearest graph nodes
immobiliare_df['node'] = immobiliare_df.apply(lambda row: ox.distance.nearest_nodes(G, row['Longitudes'], row['Latitude']), axis=1)
filtered_booking_df['node'] = filtered_booking_df.apply(lambda row: ox.distance.nearest_nodes(G, row['longitude'], row['latitude']), axis=1)

In [ ]:
# Step 4: Compute distances
bnb_node_map = defaultdict(list)
for _, row in filtered_booking_df.iterrows():
    bnb_node_map[row['node']].append({'id': row['url'], 'name': row['title']})


In [ ]:
bnb_node_map[row['node']]

In [ ]:
results = []
for _, prop in tqdm(immobiliare_df.iterrows(), total=len(immobiliare_df)):
    prop_node = prop['node']
    # Use Dijkstra to get distances from this property to all B&Bs
    distances = nx.single_source_dijkstra_path_length(G, prop_node, weight='length')
    matched_bnb = [
        (bnb_node, dist) for bnb_node, dist in distances.items() if bnb_node in bnb_node_map
    ]
    matched_bnb = sorted(matched_bnb, key=lambda x: x[1])[:10]

    for bnb_node, dist in matched_bnb:
        for bnb_info in bnb_node_map[bnb_node]:
            results.append({
                'property_id': prop['ID'],
                'bnb_id': bnb_info['id'],
                'bnb_name': bnb_info['name'],
                'distance_m': dist
            })
df_results = pd.DataFrame(results)

In [ ]:
# COMPUTE PROPERTIES FOR BOOKING B&Bs

In [ ]:
def compute_monthly_stats(apartment_df, column_name):
    """
    For each row in the DataFrame, compute the monthly occupancy rate and average price.
    
    Args:
        apartment_df (pd.DataFrame): DataFrame where each row represents an apartment
                                     and contains a list of (date, price) tuples in `column_name`.
        column_name (str): The name of the column containing the list of (date, price) tuples.

    Returns:
        pd.DataFrame: A DataFrame with multi-index (apartment index, month), and columns for
                      occupancy_rate and average_price.
    """
    results = []

    for idx, row in apartment_df.iterrows():
        date_price_list = row[column_name]
        prices = []
        monthly_stats = defaultdict(lambda: {"available_days": 0, "total_days": 0})
        for date_str, price in date_price_list:
            date = datetime.strptime(date_str, "%Y-%m-%d")
            month = date.strftime("%Y-%m")
            monthly_stats[month]["total_days"] += 1
            if price is not None:
                monthly_stats[month]["available_days"] += 1
                # Clean and parse the price
                price_value = float(price.replace("€", "").replace(",", "").replace("K", "000").strip())
                prices.append(price_value)

        for month, stats in monthly_stats.items():
            occupancy_rate = 1- (stats["available_days"] / stats["total_days"]) if stats["total_days"] > 0 else 0
            avg_price = np.percentile(list(set(prices)), 25) if prices else 0
            results.append({
                "apartment_index": idx,
                "month": month,
                "occupancy_rate": round(occupancy_rate, 2),
                "average_price": round(avg_price, 2)
            })

    return pd.DataFrame(results).set_index(['apartment_index', 'month'])


In [ ]:
filtered_booking_df_stats = compute_monthly_stats(filtered_booking_df, column_name="calendar")
filtered_booking_df_stats = filtered_booking_df_stats.reset_index()
filtered_booking_df_stats = pd.merge(filtered_booking_df_stats, filtered_booking_df[['url', 'title', 'score', 'latitude', 'longitude']], how='left', left_on='apartment_index', right_on='url')
filtered_booking_df_stats_may = filtered_booking_df_stats[filtered_booking_df_stats['month']=='2025-05']
#filtered_booking_df_stats_may = filtered_booking_df_stats_may[filtered_booking_df_stats_may['occupancy_rate']!=1.00]

In [ ]:
filtered_booking_df_stats_may

In [ ]:
# Step 1: Merge B&B metadata into results_df
# Make sure bnb_id matches df_bnb column (adjust if needed)
merged = df_results.merge(
    filtered_booking_df_stats_may[['url', 'occupancy_rate', 'average_price']],
    left_on='bnb_id',
    right_on='url',
    how='left'
)

# Step 2: Group by property and compute averages
# Group by property and compute stats
aggregated = merged.groupby('property_id').agg(
    avg_occupancy_rate=('occupancy_rate', 'mean'),
    std_occupancy_rate=('occupancy_rate', 'std'),
    avg_price_per_night=('average_price', 'mean'),
    std_price_per_night=('average_price', 'std'),
    bnb_count=('bnb_id', 'count')
).reset_index()

aggregated['expected_outcome'] = aggregated['avg_occupancy_rate'] * aggregated['avg_price_per_night'] * 30


# Assuming 'price' column in df_properties is the property price
df_properties_with_stats = immobiliare_df.merge(
    aggregated,
    left_on='ID',
    right_on='property_id',
    how='left'
)

# FIX PRICE
df_properties_with_stats['Prezzo'] = (
    df_properties_with_stats['Prezzo']
    .str.replace("€", "", regex=False)  # remove euro symbol
    .str.replace(".", "", regex=False)  # remove thousands separator
    .str.replace(",", ".", regex=False)  # optional, only if commas are decimal
    .str.strip()
    .astype(float)
)

# Standardize by property price
df_properties_with_stats['standardized_outcome'] = (
    df_properties_with_stats['expected_outcome'] / df_properties_with_stats['Prezzo']
)

In [ ]:
df_properties_with_stats[df_properties_with_stats['ID']==119842414]

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="expected_outcome",         # ROI: outcome / price
    size="expected_outcome",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("property_map_expected_monthly_outcome.html")

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="standardized_outcome",         # ROI: outcome / price
    size="expected_outcome",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("property_map_standardized_monthly_outcome.html")

In [ ]:
# FILTER B&Bs

In [ ]:
avg_prices = []
for idx, row in filtered_booking_df.iterrows():
    date_price_list = row['calendar']
    prices = []
    for date_str, price in date_price_list:
        if price is not None:
            price_value = float(price.replace("€", "").replace(",", "").replace("K", "000").strip())
            prices.append(price_value)

    avg_prices.append(np.percentile(list(set(prices)), 25) if prices else 0)

In [ ]:
filtered_booking_df['price'] = avg_prices

In [ ]:
np.percentile(filtered_booking_df['price'], 85)

In [ ]:
filtered_booking_df_filtered = filtered_booking_df[filtered_booking_df['price']<np.percentile(filtered_booking_df['price'], 85)]

# Step 4: Compute distances
results = []
bnb_node_map = defaultdict(list)
for _, row in filtered_booking_df_filtered.iterrows():
    bnb_node_map[row['node']].append({'id': row['url'], 'name': row['title']})


In [ ]:
results = []
for _, prop in tqdm(immobiliare_df.iterrows(), total=len(immobiliare_df)):
    prop_node = prop['node']
    # Use Dijkstra to get distances from this property to all B&Bs
    distances = nx.single_source_dijkstra_path_length(G, prop_node, weight='length')
    matched_bnb = [
        (bnb_node, dist) for bnb_node, dist in distances.items() if bnb_node in bnb_node_map
    ]
    matched_bnb = sorted(matched_bnb, key=lambda x: x[1])[:5]

    for bnb_node, dist in matched_bnb:
        for bnb_info in bnb_node_map[bnb_node]:
            results.append({
                'property_id': prop['ID'],
                'bnb_id': bnb_info['id'],
                'bnb_name': bnb_info['name'],
                'distance_m': dist
            })
df_results_filtered = pd.DataFrame(results)

In [ ]:
df_results_filtered = df_results_filtered[df_results_filtered['distance_m']<800]

In [ ]:
filtered_booking_df_stats = compute_monthly_stats(filtered_booking_df, column_name="calendar")
filtered_booking_df_stats = filtered_booking_df_stats.reset_index()
filtered_booking_df_stats = pd.merge(filtered_booking_df_stats, filtered_booking_df[['url', 'title', 'score', 'latitude', 'longitude']], how='left', left_on='apartment_index', right_on='url')
filtered_booking_df_stats_may = filtered_booking_df_stats[filtered_booking_df_stats['month']=='2025-05']
#filtered_booking_df_stats_may = filtered_booking_df_stats_may[filtered_booking_df_stats_may['occupancy_rate']!=1.00]

In [ ]:
# Step 1: Merge B&B metadata into results_df
# Make sure bnb_id matches df_bnb column (adjust if needed)
merged = df_results_filtered.merge(
    filtered_booking_df_stats_may[['url', 'occupancy_rate', 'average_price']],
    left_on='bnb_id',
    right_on='url',
    how='left'
)

# Step 2: Group by property and compute averages
# Group by property and compute stats
aggregated = merged.groupby('property_id').agg(
    avg_occupancy_rate=('occupancy_rate', 'mean'),
    std_occupancy_rate=('occupancy_rate', 'std'),
    avg_price_per_night=('average_price', 'mean'),
    std_price_per_night=('average_price', 'std'),
    bnb_count=('bnb_id', 'count')
).reset_index()

aggregated['expected_outcome'] = aggregated['avg_occupancy_rate'] * aggregated['avg_price_per_night'] * 30


# Assuming 'price' column in df_properties is the property price
df_properties_with_stats = immobiliare_df.merge(
    aggregated,
    left_on='ID',
    right_on='property_id',
    how='left'
)

# FIX PRICE
df_properties_with_stats['Prezzo'] = (
    df_properties_with_stats['Prezzo']
    .str.replace("€", "", regex=False)  # remove euro symbol
    .str.replace(".", "", regex=False)  # remove thousands separator
    .str.replace(",", ".", regex=False)  # optional, only if commas are decimal
    .str.strip()
    .astype(float)
)

# Standardize by property price
df_properties_with_stats['standardized_outcome'] = (
    df_properties_with_stats['expected_outcome'] / df_properties_with_stats['Prezzo']
)

In [ ]:
df_properties_with_stats = df_properties_with_stats[df_properties_with_stats['bnb_count']>0]
df_properties_with_stats

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="expected_outcome",         # ROI: outcome / price
    size="bnb_count",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("may_property_map_expected_monthly_outcome.html")

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="standardized_outcome",         # ROI: outcome / price
    size="bnb_count",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("may_property_map_standardized_monthly_outcome.html")

In [ ]:
for i, row in df_properties_with_stats.iterrows():
    print(row['ID'], row['Titolo'])

In [ ]:
df_properties_with_stats.sort_values("standardized_outcome", ascending=False)[["Titolo", "Prezzo", "avg_occupancy_rate","std_occupancy_rate","avg_price_per_night","std_price_per_night","bnb_count","expected_outcome","standardized_outcome"]]

In [ ]:
df_properties_with_stats[df_properties_with_stats['ID']==120396918]

In [ ]:
selected_property_id = 120396918
property_row = immobiliare_df[immobiliare_df['ID'] == selected_property_id].iloc[0]

# Step 2: Get nearest B&Bs
selected_results = df_results[df_results['property_id'] == selected_property_id]
bnb_ids = selected_results['bnb_id'].tolist()

# Step 3: Get B&B info with coordinates
bnb_subset = filtered_booking_df_stats_may[filtered_booking_df_stats_may['url'].isin(bnb_ids)].copy()
bnb_subset['type'] = 'B&B'
bnb_subset['name'] = bnb_subset['title']
bnb_subset['id'] = bnb_subset['url']
bnb_subset['size'] = 20

# Step 4: Add property info as one row
property_df = pd.DataFrame([{
    'id': property_row['ID'],
    'latitude': property_row['Latitude'],
    'longitude': property_row['Longitudes'],
    'name': property_row['Titolo'],
    'type': 'Property',
    'size': 25
}])

# Step 5: Concatenate for plot
plot_df = pd.concat([
    property_df[['id', 'latitude', 'longitude', 'name', 'type', 'size']],
    bnb_subset[['id', 'latitude', 'longitude', 'name', 'type', 'size', 'average_price', 'occupancy_rate', 'score']]
])

# Step 6: Plot with Plotly Express
fig = px.scatter_mapbox(
    plot_df,
    lat="latitude",
    lon="longitude",
    color="type",
    hover_name="name",
    hover_data=[
        "average_price",
        "occupancy_rate",
        "score"
    ],
    zoom=15,
    size="size",
    mapbox_style="open-street-map",
    height=600
)

fig.update_layout(title_text=f"Property ID: {selected_property_id}\tPrice: {price}\tExpected: {expected_outcome}\tROI: {standardized_outcome}")
fig.show()

In [ ]:
plot_df

In [ ]:
filtered_booking_df_stats_may

In [ ]:
bnb_subset

In [ ]:
properties_top = df_properties_with_stats[df_properties_with_stats['standardized_outcome']>0.023]['ID'].tolist()
properties_top

In [ ]:
import plotly.express as px
import pandas as pd

# Your dataframes
# df_properties: has ['id', 'latitude', 'longitude']
# df_bnb: has ['url' or 'bnb_id', 'latitude', 'longitude', 'title' or 'bnb_name']
# results_df: has ['property_id', 'bnb_id', 'bnb_name', 'distance_m']

# Step 1: Select one property
for selected_property_id in properties_top:
    stats_property = df_properties_with_stats[df_properties_with_stats['ID'] == selected_property_id].iloc[0]
    price = stats_property['Prezzo']
    expected_outcome = stats_property['expected_outcome']
    standardized_outcome = stats_property['standardized_outcome']
    property_row = immobiliare_df[immobiliare_df['ID'] == selected_property_id].iloc[0]
    
    # Step 2: Get nearest B&Bs
    selected_results = df_results[df_results['property_id'] == selected_property_id]
    bnb_ids = selected_results['bnb_id'].tolist()

    # Step 3: Get B&B info with coordinates
    bnb_subset = filtered_booking_df[filtered_booking_df['url'].isin(bnb_ids)].copy()
    bnb_subset['type'] = 'B&B'
    bnb_subset['name'] = bnb_subset['title']
    bnb_subset['id'] = bnb_subset['url']
    bnb_subset['size'] = 20

    # Step 4: Add property info as one row
    property_df = pd.DataFrame([{
        'id': property_row['ID'],
        'latitude': property_row['Latitude'],
        'longitude': property_row['Longitudes'],
        'name': property_row['Titolo'],
        'type': 'Property',
        'size': 25
    }])

    # Step 5: Concatenate for plot
    plot_df = pd.concat([
        property_df[['id', 'latitude', 'longitude', 'name', 'type', 'size']],
        bnb_subset[['id', 'latitude', 'longitude', 'name', 'type', 'size']]
    ])

    # Step 6: Plot with Plotly Express
    fig = px.scatter_mapbox(
        plot_df,
        lat="latitude",
        lon="longitude",
        color="type",
        hover_name="name",
        zoom=15,
        size="size",
        mapbox_style="open-street-map",
        height=600
    )

    fig.update_layout(title_text=f"Property ID: {selected_property_id}\tPrice: {price}\tExpected: {expected_outcome}\tROI: {standardized_outcome}")
    fig.show()

In [ ]:
# 2nd month

In [ ]:
filtered_booking_df_stats = compute_monthly_stats(filtered_booking_df, column_name="calendar")
filtered_booking_df_stats = filtered_booking_df_stats.reset_index()
filtered_booking_df_stats = pd.merge(filtered_booking_df_stats, filtered_booking_df[['url', 'title', 'score', 'latitude', 'longitude']], how='left', left_on='apartment_index', right_on='url')
filtered_booking_df_stats_may = filtered_booking_df_stats[filtered_booking_df_stats['month']=='2025-06']
#filtered_booking_df_stats_may = filtered_booking_df_stats_may[filtered_booking_df_stats_may['occupancy_rate']!=1.00]

In [ ]:
# Step 1: Merge B&B metadata into results_df
# Make sure bnb_id matches df_bnb column (adjust if needed)
merged = df_results_filtered.merge(
    filtered_booking_df_stats_may[['url', 'occupancy_rate', 'average_price']],
    left_on='bnb_id',
    right_on='url',
    how='left'
)

# Step 2: Group by property and compute averages
# Group by property and compute stats
aggregated = merged.groupby('property_id').agg(
    avg_occupancy_rate=('occupancy_rate', 'mean'),
    std_occupancy_rate=('occupancy_rate', 'std'),
    avg_price_per_night=('average_price', 'mean'),
    std_price_per_night=('average_price', 'std'),
    bnb_count=('bnb_id', 'count')
).reset_index()

aggregated['expected_outcome'] = aggregated['avg_occupancy_rate'] * aggregated['avg_price_per_night'] * 30


# Assuming 'price' column in df_properties is the property price
df_properties_with_stats = immobiliare_df.merge(
    aggregated,
    left_on='ID',
    right_on='property_id',
    how='left'
)

# FIX PRICE
df_properties_with_stats['Prezzo'] = (
    df_properties_with_stats['Prezzo']
    .str.replace("€", "", regex=False)  # remove euro symbol
    .str.replace(".", "", regex=False)  # remove thousands separator
    .str.replace(",", ".", regex=False)  # optional, only if commas are decimal
    .str.strip()
    .astype(float)
)

# Standardize by property price
df_properties_with_stats['standardized_outcome'] = (
    df_properties_with_stats['expected_outcome'] / df_properties_with_stats['Prezzo']
)

In [ ]:
df_properties_with_stats = df_properties_with_stats[df_properties_with_stats['bnb_count']>0]
df_properties_with_stats

In [ ]:
df_properties_with_stats[df_properties_with_stats['ID']==120396918]

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="expected_outcome",         # ROI: outcome / price
    size="bnb_count",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("june_property_map_expected_monthly_outcome.html")

In [ ]:
fig = px.scatter_mapbox(
    df_properties_with_stats,
    lat="Latitude",
    lon="Longitudes",
    color="standardized_outcome",         # ROI: outcome / price
    size="bnb_count",              # Total income potential
    hover_data=[
        "Titolo",
        "Prezzo",
        "bnb_count",
        "avg_occupancy_rate", "std_occupancy_rate",
        "avg_price_per_night", "std_price_per_night",
        "expected_outcome", "standardized_outcome"
    ],
    zoom=12,
    mapbox_style="open-street-map",
    height=650,
    title="Properties: ROI (color) vs. Expected Outcome (size)"
)

fig.show()

In [ ]:
fig.write_html("june_property_map_standardized_monthly_outcome.html")

In [ ]:
selected_property_id = 120396918
property_row = immobiliare_df[immobiliare_df['ID'] == selected_property_id].iloc[0]

# Step 2: Get nearest B&Bs
selected_results = df_results[df_results['property_id'] == selected_property_id]
bnb_ids = selected_results['bnb_id'].tolist()

# Step 3: Get B&B info with coordinates
bnb_subset = filtered_booking_df_stats_may[filtered_booking_df_stats_may['url'].isin(bnb_ids)].copy()
bnb_subset['type'] = 'B&B'
bnb_subset['name'] = bnb_subset['title']
bnb_subset['id'] = bnb_subset['url']
bnb_subset['size'] = 20

# Step 4: Add property info as one row
property_df = pd.DataFrame([{
    'id': property_row['ID'],
    'latitude': property_row['Latitude'],
    'longitude': property_row['Longitudes'],
    'name': property_row['Titolo'],
    'type': 'Property',
    'size': 25
}])

# Step 5: Concatenate for plot
plot_df = pd.concat([
    property_df[['id', 'latitude', 'longitude', 'name', 'type', 'size']],
    bnb_subset[['id', 'latitude', 'longitude', 'name', 'type', 'size', 'average_price', 'occupancy_rate', 'score']]
])

# Step 6: Plot with Plotly Express
fig = px.scatter_mapbox(
    plot_df,
    lat="latitude",
    lon="longitude",
    color="type",
    hover_name="name",
    hover_data=[
        "average_price",
        "occupancy_rate",
        "score"
    ],
    zoom=15,
    size="size",
    mapbox_style="open-street-map",
    height=600
)

fig.update_layout(title_text=f"Property ID: {selected_property_id}\tPrice: {price}\tExpected: {expected_outcome}\tROI: {standardized_outcome}")
fig.show()

In [ ]:
plot_df

In [ ]:
properties_top = df_properties_with_stats[df_properties_with_stats['standardized_outcome']>0.014]['ID'].tolist()
properties_top

In [ ]:
import plotly.express as px
import pandas as pd

# Your dataframes
# df_properties: has ['id', 'latitude', 'longitude']
# df_bnb: has ['url' or 'bnb_id', 'latitude', 'longitude', 'title' or 'bnb_name']
# results_df: has ['property_id', 'bnb_id', 'bnb_name', 'distance_m']

# Step 1: Select one property
for selected_property_id in properties_top:
    stats_property = df_properties_with_stats[df_properties_with_stats['ID'] == selected_property_id].iloc[0]
    price = stats_property['Prezzo']
    expected_outcome = stats_property['expected_outcome']
    standardized_outcome = stats_property['standardized_outcome']
    property_row = immobiliare_df[immobiliare_df['ID'] == selected_property_id].iloc[0]
    
    # Step 2: Get nearest B&Bs
    selected_results = df_results[df_results['property_id'] == selected_property_id]
    bnb_ids = selected_results['bnb_id'].tolist()

    # Step 3: Get B&B info with coordinates
    bnb_subset = filtered_booking_df[filtered_booking_df['url'].isin(bnb_ids)].copy()
    bnb_subset['type'] = 'B&B'
    bnb_subset['name'] = bnb_subset['title']
    bnb_subset['id'] = bnb_subset['url']
    bnb_subset['size'] = 20

    # Step 4: Add property info as one row
    property_df = pd.DataFrame([{
        'id': property_row['ID'],
        'latitude': property_row['Latitude'],
        'longitude': property_row['Longitudes'],
        'name': property_row['Titolo'],
        'type': 'Property',
        'size': 25
    }])

    # Step 5: Concatenate for plot
    plot_df = pd.concat([
        property_df[['id', 'latitude', 'longitude', 'name', 'type', 'size']],
        bnb_subset[['id', 'latitude', 'longitude', 'name', 'type', 'size']]
    ])

    # Step 6: Plot with Plotly Express
    fig = px.scatter_mapbox(
        plot_df,
        lat="latitude",
        lon="longitude",
        color="type",
        hover_name="name",
        zoom=15,
        size="size",
        mapbox_style="open-street-map",
        height=600
    )

    fig.update_layout(title_text=f"Property ID: {selected_property_id}\tPrice: {price}\tExpected: {expected_outcome}\tROI: {standardized_outcome}")
    fig.show()

In [ ]:
df_properties_with_stats

In [ ]:
df_results.head().to_markdown()

In [ ]:
df_results[df_results['distance_m']>1000]

In [ ]:
import plotly.express as px
import pandas as pd

# Your dataframes
# df_properties: has ['id', 'latitude', 'longitude']
# df_bnb: has ['url' or 'bnb_id', 'latitude', 'longitude', 'title' or 'bnb_name']
# results_df: has ['property_id', 'bnb_id', 'bnb_name', 'distance_m']

# Step 1: Select one property
selected_property_id = 117823375
property_row = immobiliare_df[immobiliare_df['ID'] == selected_property_id].iloc[0]

# Step 2: Get nearest B&Bs
selected_results = df_results[df_results['property_id'] == selected_property_id]
bnb_ids = selected_results['bnb_id'].tolist()

# Step 3: Get B&B info with coordinates
bnb_subset = filtered_booking_df[filtered_booking_df['url'].isin(bnb_ids)].copy()
bnb_subset['type'] = 'B&B'
bnb_subset['name'] = bnb_subset['title']
bnb_subset['id'] = bnb_subset['url']
bnb_subset['size'] = 20

# Step 4: Add property info as one row
property_df = pd.DataFrame([{
    'id': property_row['ID'],
    'latitude': property_row['Latitude'],
    'longitude': property_row['Longitudes'],
    'name': 'Property',
    'type': 'Property',
    'size': 25
}])

# Step 5: Concatenate for plot
plot_df = pd.concat([
    property_df[['id', 'latitude', 'longitude', 'name', 'type', 'size']],
    bnb_subset[['id', 'latitude', 'longitude', 'name', 'type', 'size']]
])

# Step 6: Plot with Plotly Express
fig = px.scatter_mapbox(
    plot_df,
    lat="latitude",
    lon="longitude",
    color="type",
    hover_name="name",
    zoom=15,
    size="size",
    mapbox_style="open-street-map",
    height=600
)

fig.update_layout(title_text=f"Property ID {selected_property_id} and Nearby B&Bs")
fig.show()